# 05 — Data quality and audit
Runs basic completeness, uniqueness, and referential-integrity checks, then writes the results to Delta audit tables.

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime
import uuid

run_id = str(uuid.uuid4())
run_started = datetime.utcnow()

checks = []

def add_check(name, passed, observed, expectation):
    checks.append((run_id, name, bool(passed), str(observed), expectation, datetime.utcnow()))

# Primary key checks
duplicate_orders = (spark.table("silver.orders").groupBy("order_id").count().filter("count > 1").count())
add_check("silver_orders_unique_order_id", duplicate_orders == 0, duplicate_orders, "0 duplicates")

duplicate_items = (spark.table("silver.order_items")
                   .groupBy("order_id", "order_item_id").count()
                   .filter("count > 1").count())
add_check("silver_order_items_unique_grain", duplicate_items == 0, duplicate_items, "0 duplicate order-item keys")

# Required key null checks
null_order_ids = spark.table("silver.orders").filter(F.col("order_id").isNull()).count()
add_check("silver_orders_order_id_not_null", null_order_ids == 0, null_order_ids, "0 null order_id values")

# Fact-to-dimension integrity
orphan_customers = (spark.table("gold.fact_order_item").alias("f")
    .filter(F.col("f.customer_key").isNotNull())
    .join(spark.table("gold.dim_customer").select("customer_key").alias("d"),
          "customer_key", "left_anti").count())
add_check("fact_customer_key_referential_integrity", orphan_customers == 0, orphan_customers, "0 orphan customer keys")

# SCD current row uniqueness
multiple_current = (spark.table("silver.customer_history")
    .filter("is_current = true").groupBy("customer_id").count()
    .filter("count > 1").count())
add_check("scd2_one_current_version", multiple_current == 0, multiple_current, "0 customers with >1 current row")

dq_schema = ["run_id", "check_name", "passed", "observed_value", "expectation", "checked_at"]
dq_df = spark.createDataFrame(checks, dq_schema)
(dq_df.write.format("delta").mode("append").saveAsTable("ops.dq_results"))

run_finished = datetime.utcnow()
audit_row = [(run_id, "05_data_quality_and_audit", run_started, run_finished,
              len(checks), sum(1 for c in checks if c[2]), sum(1 for c in checks if not c[2]))]
audit_schema = ["run_id", "notebook_name", "started_at", "finished_at", "checks_total", "checks_passed", "checks_failed"]
(spark.createDataFrame(audit_row, audit_schema)
 .write.format("delta").mode("append").saveAsTable("ops.audit_run"))

display(dq_df.orderBy("passed", "check_name"))
assert all(c[2] for c in checks), "One or more data quality checks failed. Review ops.dq_results."
print("Data quality checks passed:", sum(1 for c in checks if c[2]), "/", len(checks))

## Operational tables
- `ops.dq_results`: one row per data-quality check execution.
- `ops.audit_run`: summary for this notebook's validation run.

Extend audit logging to all notebooks when converting this portfolio into a production implementation.